In [ ]:
!pip install -q pandas numpy scikit-learn sentence-transformers plotly openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/Insurance_complaints__All_data1.csv"
)

print(df.shape)
df.head()

(299677, 17)


,Complaint number,Complaint filed against,Complaint filed by,Reason complaint filed,Confirmed complaint,How resolved,Received date,Closed date,Complaint type,Coverage type,Coverage level,Others involved,Respondent ID,Respondent Role,Respondent type,Complainant type,Keywords
0,1,METROPOLITAN LIFE INSURANCE COMPANY,Relative,Cust Service Claim Handling,No,Other,06-12-2012,07/25/2012,"Life, Accident and Health",Life & Annuity,Individual Life,NaN,13191,Ins Co - Licensed/Active,Organization,INDV,NaN
1,2,AETNA LIFE INSURANCE COMPANY,Provider,Delays (Claims Handling),No,Information Furnished,06/21/2012,08-01-2012,"Life, Accident and Health",Accident and Health,Group A&H,Insured,245,Ins Co - Licensed/Active,Organization,ORG,NaN
2,3,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Provider,Denial Of Claim,No,Other,06-11-2012,07/30/2012,"Life, Accident and Health",Accident and Health,Group A&H,NaN,10047,Ins Co - Licensed/Active,Organization,ORG,NaN
3,4,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Provider,Denial Of Claim,No,Other,06/28/2012,07/30/2012,"Life, Accident and Health",Accident and Health,Group A&H,NaN,10047,Ins Co - Licensed/Active,Organization,ORG,NaN
4,5,"CHARTER OAK FIRE INSURANCE COMPANY, THE",Insured,Unsatisfactory Settle/Offer,No,Contract Language/Legal Issue; Question of Fact,06/13/2012,07/17/2012,Property and Casualty,Automobile,Individual Private Pass,NaN,2918,Ins Co - Licensed/Active,Organization,INDV,2012 NORTH TEXAS TORNADOES; ADJUSTER'S HANDLIN...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299677 entries, 0 to 299676
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   Complaint number         299677 non-null  int64 
 1   Complaint filed against  299677 non-null  object
 2   Complaint filed by       299673 non-null  object
 3   Reason complaint filed   299660 non-null  object
 4   Confirmed complaint      299677 non-null  object
 5   How resolved             298548 non-null  object
 6   Received date            299677 non-null  object
 7   Closed date              299677 non-null  object
 8   Complaint type           299676 non-null  object
 9   Coverage type            299635 non-null  object
 10  Coverage level           299635 non-null  object
 11  Others involved          270707 non-null  object
 12  Respondent ID            299677 non-null  int64 
 13  Respondent Role          299675 non-null  object
 14  Respondent type     

In [ ]:
df.isnull().sum()

,0
Complaint number,0
Complaint filed against,0
Complaint filed by,4
Reason complaint filed,17
Confirmed complaint,0
How resolved,1129
Received date,0
Closed date,0
Complaint type,1
Coverage type,42


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["Reason complaint filed"].value_counts().head(20)

,count
Reason complaint filed,
Denial Of Claim,27968
Delays (Claims Handling),27218
Cust Service Claim Handling; Delays (Claims Handling),24828
Unsatisfactory Settle/Offer,22603
Balance Billing; Unsatisfactory Settle/Offer,17611
Cust Service Claim Handling; Delays (Claims Handling); Unsatisfactory Settle/Offer,11560
Delays (Claims Handling); Unsatisfactory Settle/Offer,11416
Delays (Policyholder Service),7132
Cust Service Claim Handling; Unsatisfactory Settle/Offer,7119


In [ ]:
columns = [

"Complaint filed against",

"Reason complaint filed",

"How resolved",

"Coverage type",

"Coverage level",

"Keywords",

"Received date",

"Closed date"

]

df = df[columns]

In [ ]:
df.fillna("", inplace=True)

In [ ]:
df.head()

,Complaint filed against,Reason complaint filed,How resolved,Coverage type,Coverage level,Keywords,Received date,Closed date
0,METROPOLITAN LIFE INSURANCE COMPANY,Cust Service Claim Handling,Other,Life & Annuity,Individual Life,,06-12-2012,07/25/2012
1,AETNA LIFE INSURANCE COMPANY,Delays (Claims Handling),Information Furnished,Accident and Health,Group A&H,,06/21/2012,08-01-2012
2,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Denial Of Claim,Other,Accident and Health,Group A&H,,06-11-2012,07/30/2012
3,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Denial Of Claim,Other,Accident and Health,Group A&H,,06/28/2012,07/30/2012
4,"CHARTER OAK FIRE INSURANCE COMPANY, THE",Unsatisfactory Settle/Offer,Contract Language/Legal Issue; Question of Fact,Automobile,Individual Private Pass,2012 NORTH TEXAS TORNADOES; ADJUSTER'S HANDLIN...,06/13/2012,07/17/2012


In [ ]:
df["Received date"] = pd.to_datetime(
    df["Received date"],
    errors="coerce"
)

df["Closed date"] = pd.to_datetime(
    df["Closed date"],
    errors="coerce"
)

In [ ]:
df["Resolution_Days"] = (

df["Closed date"]

-

df["Received date"]

).dt.days

In [ ]:
df["Complaint_Text"] = (

"Reason: "

+ df["Reason complaint filed"].astype(str)

+ ". Coverage: "

+ df["Coverage type"].astype(str)

+ ". Keywords: "

+ df["Keywords"].astype(str)

+ ". Resolution: "

+ df["How resolved"].astype(str)

)

In [ ]:
df.to_csv(

"/content/drive/MyDrive/cleaned_insurance_data.csv",

index=False

)

In [ ]:
import plotly.express as px

reason = (

df["Reason complaint filed"]

.value_counts()

.head(10)

)

fig = px.bar(

x=reason.index,

y=reason.values,

title="Top Complaint Reasons"

)

fig.show()

In [ ]:
coverage = (

df["Coverage type"]

.value_counts()

.head(10)

)

fig = px.pie(

values=coverage.values,

names=coverage.index,

title="Coverage Types"

)

fig.show()

In [ ]:
company = (

df["Complaint filed against"]

.value_counts()

.head(10)

)

fig = px.bar(

x=company.index,

y=company.values,

title="Top Companies"

)

fig.show()

In [ ]:
fig = px.histogram(

df,

x="Resolution_Days",

nbins=40,

title="Resolution Time"

)

fig.show()

In [ ]:
df.to_csv(

"/content/drive/MyDrive/eda_completed.csv",

index=False

)

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/eda_completed.csv"
)

print(df.shape)
df.head()

(299677, 10)


,Complaint filed against,Reason complaint filed,How resolved,Coverage type,Coverage level,Keywords,Received date,Closed date,Resolution_Days,Complaint_Text
0,METROPOLITAN LIFE INSURANCE COMPANY,Cust Service Claim Handling,Other,Life & Annuity,Individual Life,NaN,2012-06-12,2012-07-25,43.0,Reason: Cust Service Claim Handling. Coverage:...
1,AETNA LIFE INSURANCE COMPANY,Delays (Claims Handling),Information Furnished,Accident and Health,Group A&H,NaN,NaN,NaN,NaN,Reason: Delays (Claims Handling). Coverage: Ac...
2,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Denial Of Claim,Other,Accident and Health,Group A&H,NaN,2012-06-11,2012-07-30,49.0,Reason: Denial Of Claim. Coverage: Accident an...
3,"BLUE CROSS AND BLUE SHIELD OF TEXAS, A DIVISIO...",Denial Of Claim,Other,Accident and Health,Group A&H,NaN,NaN,2012-07-30,NaN,Reason: Denial Of Claim. Coverage: Accident an...
4,"CHARTER OAK FIRE INSURANCE COMPANY, THE",Unsatisfactory Settle/Offer,Contract Language/Legal Issue; Question of Fact,Automobile,Individual Private Pass,2012 NORTH TEXAS TORNADOES; ADJUSTER'S HANDLIN...,NaN,2012-07-17,NaN,Reason: Unsatisfactory Settle/Offer. Coverage:...


In [ ]:
sample_df = df.sample(
    n=20000,
    random_state=42
).reset_index(drop=True)

print(sample_df.shape)

(20000, 10)


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(
    sample_df["Complaint_Text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)

(20000, 384)


In [ ]:
import numpy as np

np.save(
    "/content/drive/MyDrive/complaint_embeddings.npy",
    embeddings
)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=10,
    random_state=42
)

sample_df["Cluster"] = kmeans.fit_predict(embeddings)

In [ ]:
sample_df["Cluster"].value_counts().sort_index()

,count
Cluster,
0,1100
1,1416
2,2058
3,1673
4,2388
5,4233
6,2273
7,2834
8,1034


In [ ]:
for cluster in sorted(sample_df["Cluster"].unique()):

    print("=" * 60)
    print(f"CLUSTER {cluster}")

    display(
        sample_df[
            sample_df["Cluster"] == cluster
        ]["Reason complaint filed"]
        .value_counts()
        .head(10)
    )

CLUSTER 0


,count
Reason complaint filed,
Balance Billing; Unsatisfactory Settle/Offer,995
Balance Billing; Medical Necessity; Unsatisfactory Settle/Offer,43
Balance Billing,16
Balance Billing; PPACA-Out-Of-Ntwk Emerg Care; Unsatisfactory Settle/Offer,9
Unsatisfactory Settle/Offer,6
Balance Billing; Medical Necessity; Provider Ntwk Disclosure; Unsatisfactory Settle/Offer,5
Balance Billing; Provider Ntwk Disclosure; Unsatisfactory Settle/Offer,3
Balance Billing; Maternity and Newborn Care; Medical Necessity; Unsatisfactory Settle/Offer,2
Denial Of Claim,2


CLUSTER 1


,count
Reason complaint filed,
Cust Service Claim Handling; Delays (Claims Handling),403
Cust Service Claim Handling; Delays (Claims Handling); Unsatisfactory Settle/Offer,154
Cust Service Claim Handling; Unsatisfactory Settle/Offer,151
Cust Service Claim Handling; Denial Of Claim,132
Cust Service Claim Handling,115
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim,66
Cust Service Claim Handling; Denial Of Claim; Unsatisfactory Settle/Offer,40
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim; Unsatisfactory Settle/Offer,21
Cust Service Claim Handling; Delays (Policyholder Service),14


CLUSTER 2


...

,count
Reason complaint filed,
Cust Service Claim Handling; Delays (Claims Handling),1219
Cust Service Claim Handling; Delays (Claims Handling); Unsatisfactory Settle/Offer,617
Cust Service Claim Handling; Unsatisfactory Settle/Offer,325
Cust Service Claim Handling; Denial Of Claim,143
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim,113
Cust Service Claim Handling,104
Cust Service Claim Handling; Delays (Claims Handling); Liability Dispute,51
Cust Service Claim Handling; Liability Dispute,41
Cust Service Claim Handling; Denial Of Claim; Liability Dispute,33


CLUSTER 8


,count
Reason complaint filed,
Delays (Policyholder Service),135
Cash Value; Delays (Policyholder Service),90
Cash Value,64
Delays (Claims Handling),52
Misrepresentation,51
Other,41
Refund Of Premium,41
Unauthorized Acts,35
Denial Of Claim,34


CLUSTER 9


,count
Reason complaint filed,
Delays (Policyholder Service),246
Delays (Policyholder Service); Refund Of Premium,164
Cust Srvc Policy Holder Srvcs; Delays (Policyholder Service),64
Cust Srvc Policy Holder Srvcs,57
Cust Srvc Policy Holder Srvcs; Refund Of Premium,44
Cust Srvc Policy Holder Srvcs; Delays (Policyholder Service); Refund Of Premium,44
Cancellation; Delays (Policyholder Service); Refund Of Premium,20
Delays (Policyholder Service); Premium and Rating,17
Cancellation; Delays (Policyholder Service),16


In [ ]:
sample_df.to_csv(
    "/content/drive/MyDrive/clustered_insurance_data.csv",
    index=False
)

In [ ]:
import joblib

joblib.dump(
    kmeans,
    "/content/drive/MyDrive/kmeans_model.pkl"
)

['/content/drive/MyDrive/kmeans_model.pkl']

In [ ]:
sample_df["Cluster"].value_counts().sort_index()

,count
Cluster,
0,1100
1,1416
2,2058
3,1673
4,2388
5,4233
6,2273
7,2834
8,1034


In [ ]:
for cluster in sorted(sample_df["Cluster"].unique()):

    print("=" * 80)
    print(f"CLUSTER {cluster}")

    display(
        sample_df[
            sample_df["Cluster"] == cluster
        ][
            "Reason complaint filed"
        ]
        .value_counts()
        .head(10)
    )

CLUSTER 0


,count
Reason complaint filed,
Balance Billing; Unsatisfactory Settle/Offer,995
Balance Billing; Medical Necessity; Unsatisfactory Settle/Offer,43
Balance Billing,16
Balance Billing; PPACA-Out-Of-Ntwk Emerg Care; Unsatisfactory Settle/Offer,9
Unsatisfactory Settle/Offer,6
Balance Billing; Medical Necessity; Provider Ntwk Disclosure; Unsatisfactory Settle/Offer,5
Balance Billing; Provider Ntwk Disclosure; Unsatisfactory Settle/Offer,3
Balance Billing; Maternity and Newborn Care; Medical Necessity; Unsatisfactory Settle/Offer,2
Denial Of Claim,2


CLUSTER 1


,count
Reason complaint filed,
Cust Service Claim Handling; Delays (Claims Handling),403
Cust Service Claim Handling; Delays (Claims Handling); Unsatisfactory Settle/Offer,154
Cust Service Claim Handling; Unsatisfactory Settle/Offer,151
Cust Service Claim Handling; Denial Of Claim,132
Cust Service Claim Handling,115
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim,66
Cust Service Claim Handling; Denial Of Claim; Unsatisfactory Settle/Offer,40
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim; Unsatisfactory Settle/Offer,21
Cust Service Claim Handling; Delays (Policyholder Service),14


CLUSTER 2


...

,count
Reason complaint filed,
Cust Service Claim Handling; Delays (Claims Handling),1219
Cust Service Claim Handling; Delays (Claims Handling); Unsatisfactory Settle/Offer,617
Cust Service Claim Handling; Unsatisfactory Settle/Offer,325
Cust Service Claim Handling; Denial Of Claim,143
Cust Service Claim Handling; Delays (Claims Handling); Denial Of Claim,113
Cust Service Claim Handling,104
Cust Service Claim Handling; Delays (Claims Handling); Liability Dispute,51
Cust Service Claim Handling; Liability Dispute,41
Cust Service Claim Handling; Denial Of Claim; Liability Dispute,33


CLUSTER 8


,count
Reason complaint filed,
Delays (Policyholder Service),135
Cash Value; Delays (Policyholder Service),90
Cash Value,64
Delays (Claims Handling),52
Misrepresentation,51
Other,41
Refund Of Premium,41
Unauthorized Acts,35
Denial Of Claim,34


CLUSTER 9


,count
Reason complaint filed,
Delays (Policyholder Service),246
Delays (Policyholder Service); Refund Of Premium,164
Cust Srvc Policy Holder Srvcs; Delays (Policyholder Service),64
Cust Srvc Policy Holder Srvcs,57
Cust Srvc Policy Holder Srvcs; Refund Of Premium,44
Cust Srvc Policy Holder Srvcs; Delays (Policyholder Service); Refund Of Premium,44
Cancellation; Delays (Policyholder Service); Refund Of Premium,20
Delays (Policyholder Service); Premium and Rating,17
Cancellation; Delays (Policyholder Service),16


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="API_key here",
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
root_cause_results = []

for cluster_id in sorted(sample_df["Cluster"].unique()):

    reasons = (
        sample_df[
            sample_df["Cluster"] == cluster_id
        ]["Reason complaint filed"]
        .dropna()
        .value_counts()
        .head(20)
        .index
        .tolist()
    )

    complaint_text = "\n".join(reasons)

    prompt = f"""
You are an Insurance Quality Analyst.

Below are the most common complaint reasons belonging to ONE complaint cluster.

Complaint Reasons:

{complaint_text}

Tasks:

1. Give a short Root Cause Name (3-5 words).
2. Explain the Business Impact (4-5 lines).
3. Recommend 5 Process Improvements.
4. Give a Severity (High / Medium / Low).

Return the answer in this format:

Root Cause:
Business Impact:
Recommendations:
Severity:
"""

    response = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    root_cause_results.append({
        "Cluster": cluster_id,
        "Analysis": response.choices[0].message.content
    })

print("Completed!")

Completed!


In [ ]:
import pandas as pd

results_df = pd.DataFrame(root_cause_results)

results_df.to_csv(
    "/content/drive/MyDrive/root_cause_analysis.csv",
    index=False
)

results_df

,Cluster,Analysis
0,0,Root Cause: Balance Billing Issues\n\nBusiness...
1,1,Root Cause: Claim Handling Inefficiencies\n\nB...
2,2,Root Cause: Claims Handling Issues\n\nBusiness...
3,3,Root Cause: Claims Handling Issues\n\nBusiness...
4,4,Root Cause: Claims Handling Issues\n\nBusiness...
5,5,Root Cause: Claims Handling Issues\n\nBusiness...
6,6,Root Cause: Claims Handling Issues\n\nBusiness...
7,7,Root Cause: Inefficient Claims Handling\n\nBus...
8,8,Root Cause: Poor Policyholder Service\n\nBusin...
9,9,Root Cause: Policyholder Service Delays\n\nBus...


In [ ]:
cluster_names = {

0: "Balance Billing & Settlement",

1: "Customer Service & Claims Delay",

2: "General Claims Management",

3: "Policy Administration Issues",

4: "Claims Processing Delays",

5: "Claims Settlement Disputes",

6: "Claims & Policy Violations",

7: "Customer Service Operations",

8: "Policy Servicing Issues",

9: "Policyholder Service Delays"

}

In [ ]:
sample_df["Root_Cause"] = (
    sample_df["Cluster"]
    .map(cluster_names)
)

In [ ]:
sample_df.to_csv(
    "/content/drive/MyDrive/final_insurance_rootcause.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
